# Week 03 — Monte Carlo Methods · Workbook
Eight graded exercises. References in `project/solutions.py`.

**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
import matplotlib.pyplot as plt
from project import solutions as S
from project.blackjack import BlackjackEnv
from project.recruitment import RecruitmentMDP
print('setup ok')

## Exercise 1 (★) — First-visit returns
Given a list of (state, action, reward) tuples, return a dict mapping (s, a) to the list of first-visit returns.

> **Hint**: compute and store returns backwards in $O(T)$, then scan the trajectory **forwards**, keeping the first occurrence of each (s,a). A backward `seen` check would select the last visit.

In [ ]:
def exercise_1_first_visit_returns(trajectory, gamma):
    # YOUR CODE HERE
    return S.exercise_1_first_visit_returns(trajectory, gamma)


In [ ]:
traj = [(0, 0, 0.0), (1, 1, 0.0), (2, 0, 1.0)]
out = exercise_1_first_visit_returns(traj, gamma=1.0)
assert out[(0,0)] == [1.0] and out[(1,1)] == [1.0] and out[(2,0)] == [1.0]
traj2 = [(0,0,1.0), (0,0,1.0), (1,0,0.0)]
out2 = exercise_1_first_visit_returns(traj2, gamma=1.0)
assert len(out2[(0,0)]) == 1
print('Exercise 1: OK')

## Exercise 2 (★) — Every-visit MC prediction
Evaluate the *always-stick* policy on Blackjack by every-visit MC over 3000 episodes.

> **Hint**: walk every episode and increment the running mean at every visit to a state.

In [ ]:
def exercise_2_every_visit_mc(env, policy, n_episodes=3000, gamma=1.0, seed=0):
    # YOUR CODE HERE
    return S.exercise_2_every_visit_mc(env, policy, n_episodes=n_episodes, gamma=gamma, seed=seed)


In [ ]:
env = BlackjackEnv(seed=0)
V = exercise_2_every_visit_mc(env, lambda obs: 0, n_episodes=3000, gamma=1.0, seed=0)
vs = np.array(list(V.values()))
print(f'Mean V under always-stick: {vs.mean():.3f} (expected < 0)')
assert vs.mean() < 0.0

## Exercise 3 (★★) — On-policy MC control with eps-soft
Run MC control with $\epsilon=0.1$ on Blackjack for 20k episodes. Plot the rolling-mean return.

> **Hint**: pull from `S.exercise_3_blackjack_mc_control`. Inspect the policy `pi[(s, d, u)]` for various states; you should learn to stand on 20.

In [ ]:
def exercise_3_blackjack_mc_control(n_episodes=20_000, eps=0.1, seed=0):
    # YOUR CODE HERE
    return S.exercise_3_blackjack_mc_control(n_episodes=n_episodes, eps=eps, seed=seed)


In [ ]:
Q, pi, rets = exercise_3_blackjack_mc_control(n_episodes=15_000, eps=0.1, seed=0)
win = 500
sm = np.convolve(rets, np.ones(win)/win, mode='valid')
fig, ax = plt.subplots(figsize=(5,3))
ax.plot(sm, color=RL_PALETTE['blue'])
ax.set_xlabel('episode'); ax.set_ylabel(f'rolling mean (w={win})');
ax.set_title('MC control on Blackjack');
plt.show()
print(f'mean of final 5k = {np.mean(rets[-5000:]):.3f}')
assert np.mean(rets[-5000:]) > -0.20

## Exercise 4 (★★) — Ordinary importance sampling (OIS)
Given logs of (reward, target_prob(a|x), logged_prob(a|x)), implement OIS.

> **Hint**: $w_i = \pi(a_i|x_i)/\pi_b(a_i|x_i)$; $\hat V = \frac{1}{n}\sum w_i r_i$.

In [ ]:
def exercise_4_ordinary_is(rewards, target_prob, logged_prob):
    # YOUR CODE HERE
    return S.exercise_4_ordinary_is(rewards, target_prob, logged_prob)


In [ ]:
v = exercise_4_ordinary_is([1.0, 0.0, 1.0], [0.9, 0.1, 0.5], [0.5, 0.5, 0.5])
assert abs(v - 2.8/3) < 1e-10
print('Exercise 4: OK')

## Exercise 5 (★★) — Weighted importance sampling (WIS)
Same inputs; $\hat V_{WIS} = \sum w_i r_i / \sum w_i$.

> **Hint**: same as Ex.4 but divide by $\sum w_i$ instead of $n$. Generally biased in finite samples; often lower variance, but there is no universal variance ordering.

In [ ]:
def exercise_5_weighted_is(rewards, target_prob, logged_prob):
    # YOUR CODE HERE
    return S.exercise_5_weighted_is(rewards, target_prob, logged_prob)


In [ ]:
v = exercise_5_weighted_is([1.0, 0.0, 1.0], [0.9, 0.1, 0.5], [0.5, 0.5, 0.5])
print(f'WIS = {v:.4f}')
assert abs(v - 2.8/3.0) < 1e-10

## Exercise 6 (★★) — Variance comparison OIS vs WIS
Run both estimators on 120 independent trials of a 2-armed bandit with $\pi=[0.9, 0.1]$, $\pi_b=[0.5, 0.5]$. WIS should have visibly lower variance.

> **Hint**: just call your Ex. 4 and Ex. 5 implementations repeatedly with fresh seeds.

In [ ]:
def exercise_6_compare_is_wis_variance(n_trials=120, n_logs=120, seed=0):
    # YOUR CODE HERE
    return S.exercise_6_compare_is_wis_variance(n_trials=n_trials, n_logs=n_logs, seed=seed)


In [ ]:
var_ois, var_wis = exercise_6_compare_is_wis_variance()
print(f'Var(OIS)={var_ois:.4f}; Var(WIS)={var_wis:.4f}')
assert var_wis < var_ois

## Exercise 7 (★★★, applied — *meaningful objective*, matching) — Off-policy MC on the recruitment funnel
Estimate the value of the *always advance* target policy using logs from a uniform behaviour. Compare OIS, WIS, and Monte-Carlo oracle estimates.

> **Hint**: trajectory-level $\rho$ is $2^T$ on an all-advance length-$T$ path and zero on any path that rejects; the matching path is exponentially rare under behaviour (target probability per step is 1 for advance, 0 for reject; behaviour is 1/2 per step). Per-decision IS is an alternative that can reduce variance; compare it empirically.

In [ ]:
def exercise_7_offpolicy_recruitment(n_episodes=4000, seed=0):
    # YOUR CODE HERE
    return S.exercise_7_offpolicy_recruitment(n_episodes=n_episodes, seed=seed)


In [ ]:
v_ois, v_wis, v_oracle = exercise_7_offpolicy_recruitment(n_episodes=4000, seed=0)
print(f'OIS = {v_ois:.3f}; WIS = {v_wis:.3f}; oracle = {v_oracle:.3f}')
# Sanity: both estimators are within a reasonable error of oracle (variance is high for OIS)
assert abs(v_wis - v_oracle) < 0.15

## Exercise 8 (★★) — IS variance upper bound
Verify empirically that $\mathrm{Var}(w\cdot r) \le w_{\max}^2 \cdot \mathbb{E}[r^2]$.

> **Hint**: this is a one-line consequence of $|w r| \le w_{\max}|r|$ and $\Var(X) \le \E[X^2]$.

In [ ]:
def exercise_8_is_variance_bound(p_target, p_logged, returns):
    # YOUR CODE HERE
    return S.exercise_8_is_variance_bound(p_target, p_logged, returns)


In [ ]:
rng = np.random.default_rng(0)
p_t = rng.uniform(0.05, 0.95, 200); p_b = rng.uniform(0.05, 0.95, 200); r = rng.standard_normal(200)
v, ub = exercise_8_is_variance_bound(p_t, p_b, r)
print(f'Var = {v:.4f};  bound = {ub:.4f}')
assert v <= ub + 1e-9

**Wrap-up discussion (your turn).** When would you prefer first-visit MC to every-visit? In what production setting would you reach for *doubly-robust* OPE instead of WIS? Sketch a one-paragraph protocol for *gating* a candidate matching policy by its WIS lower bound before rolling it out.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — First-visit returns</summary>

> **Hint**: compute and store returns backwards in $O(T)$, then scan the trajectory **forwards**, keeping the first occurrence of each (s,a). A backward `seen` check would select the last visit.

</details>

<details>
<summary>Exercise 2 (★) — Every-visit MC prediction</summary>

> **Hint**: walk every episode and increment the running mean at every visit to a state.

</details>

<details>
<summary>Exercise 3 (★★) — On-policy MC control with eps-soft</summary>

> **Hint**: pull from `S.exercise_3_blackjack_mc_control`. Inspect the policy `pi[(s, d, u)]` for various states; you should learn to stand on 20.

</details>

<details>
<summary>Exercise 4 (★★) — Ordinary importance sampling (OIS)</summary>

> **Hint**: $w_i = \pi(a_i|x_i)/\pi_b(a_i|x_i)$; $\hat V = \frac{1}{n}\sum w_i r_i$.

</details>

<details>
<summary>Exercise 5 (★★) — Weighted importance sampling (WIS)</summary>

> **Hint**: same as Ex.4 but divide by $\sum w_i$ instead of $n$. Generally biased in finite samples; often lower variance, but there is no universal variance ordering.

</details>

<details>
<summary>Exercise 6 (★★) — Variance comparison OIS vs WIS</summary>

> **Hint**: just call your Ex. 4 and Ex. 5 implementations repeatedly with fresh seeds.

</details>

<details>
<summary>Exercise 7 (★★★, applied — *meaningful objective*, matching) — Off-policy MC on the recruitment funnel</summary>

> **Hint**: trajectory-level $\rho$ is $2^T$ on an all-advance length-$T$ path and zero on any path that rejects; the matching path is exponentially rare under behaviour (target probability per step is 1 for advance, 0 for reject; behaviour is 1/2 per step). Per-decision IS is an alternative that can reduce variance; compare it empirically.

</details>

<details>
<summary>Exercise 8 (★★) — IS variance upper bound</summary>

> **Hint**: this is a one-line consequence of $|w r| \le w_{\max}|r|$ and $\Var(X) \le \E[X^2]$.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
